In [ ]:
!pip install -q --force-reinstall \
  "numpy==1.26.4" \
  "scipy==1.13.1" \
  "onnxruntime==1.17.1" \
  "rembg==2.0.59"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.6/64.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.3/323.3

In [ ]:
import onnxruntime as ort
import os, cv2, random, numpy as np, json
from glob import glob
from tqdm import tqdm
from rembg import remove
import matplotlib.pyplot as plt

# -------------- CONFIG --------------
TRAIN_ROOT       = "/kaggle/input/zalo-video/train"      # thư mục train (chứa annotations + samples)
TRAIN_SAMPLES    = os.path.join(TRAIN_ROOT, "samples")   # backgrounds from train
ANNOTATIONS_FILE = os.path.join(TRAIN_ROOT, "annotations", "annotations.json")

OUT_BASE = "/kaggle/working/synth_train_rembg"   # output base

SYNTH_PER_SAMPLE = 700    # synthetic images per object type
MIN_SCALE = 0.033    # ~ 23 px
MAX_SCALE = 0.16   # ~ 92 px
ROT_DEG   = 20
BLUR_PROB = 0.18   # Gaussian blur giống bản đầu
SEED      = 42
TEST_MODE = False
TEST_MAX_SAMPLES = 2

# params cho paste + mask
ALPHA_THR     = 0.08
EROSION_ITER  = 0
CENTER_MARGIN = 0.12
BIAS_EXP      = 0.7      # <1 → bias về scale lớn (object nhìn to hơn)
DEBUG_SAVE_ALPHA = False

# Occlusion + edge cut-off
OCCLUSION_PROB  = 0.15   # ~15% sample có occlusion / edge
EDGE_PROB       = 0.6    # trong 15% đó, tỉ lệ dùng edge-cut vs occlusion
OCCLUDE_AREA_RANGE = (0.10, 0.30)  # che 10–30% bbox
# -------------------------------------

random.seed(SEED)
np.random.seed(SEED)
os.makedirs(OUT_BASE, exist_ok=True)

# ------------ Extra photometric (KHÔNG có motion blur) ------------

def random_global_illumination(img,
                               p=0.85,
                               alpha_range=(0.75, 1.35),
                               beta_range=(-35, 35),
                               gamma_range=(0.7, 1.5),
                               noise_std=4.0):
    """
    Jitter global brightness/contrast + gamma + noise nhẹ.
    - alpha: contrast
    - beta: brightness
    - gamma: gamma correction
    """
    if random.random() > p:
        return img

    img_f = img.astype(np.float32)

    # random contrast + brightness
    alpha = random.uniform(*alpha_range)
    beta  = random.uniform(*beta_range)
    img_f = img_f * alpha + beta

    # gamma với xác suất 0.5
    if random.random() < 0.5:
        gamma = random.uniform(*gamma_range)
        img_f = np.power(np.clip(img_f, 0, 255) / 255.0, gamma) * 255.0

    # noise Gaussian nhẹ
    if noise_std > 0 and random.random() < 0.5:
        noise = np.random.randn(*img_f.shape).astype(np.float32) * noise_std
        img_f = img_f + noise

    img_f = np.clip(img_f, 0, 255)
    return img_f.astype(np.uint8)

# ------------ Helper: build forbidden frames map ------------
def build_forbidden_frames_map(annotations_file):
    """
    Đọc annotations.json của TRAIN, trả về:
        forbidden[video_id] = set(frame_idx) có object
    để KHÔNG lấy các frame này làm background.
    """
    if not os.path.exists(annotations_file):
        print(f"[WARN] annotations.json not found at {annotations_file} → không filter frame background.")
        return {}

    with open(annotations_file, "r") as f:
        data = json.load(f)

    forbidden = {}
    for rec in data:
        vid = rec.get("video_id")
        if not vid:
            continue
        frames = set()
        for interval in rec.get("annotations", []):
            for b in interval.get("bboxes", []):
                try:
                    fr = int(b["frame"])
                    frames.add(fr)
                except Exception:
                    continue
        if len(frames) > 0:
            forbidden[vid] = frames

    print(f"[INFO] Forbidden frame map cho {len(forbidden)} video.")
    return forbidden

# ------------ Helper: unique ref samples (Backpack_0/1, Jacket_0/1, ...) ------------
def collect_unique_ref_samples(root_dir):
    """
    Lấy object_images nhưng chỉ 1 folder cho mỗi loại:
        Backpack_0 + Backpack_1 → chỉ dùng Backpack_0
        Jacket_0 + Jacket_1     → chỉ dùng Jacket_0
        ...
    """
    samples = sorted([
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d))
    ])

    base_used   = set()
    sample_refs = {}
    for s in samples:
        base_name = s.split("_")[0]  # Backpack_0 -> Backpack
        if base_name in base_used:
            continue
        obj_dir = os.path.join(root_dir, s, "object_images")
        if not os.path.isdir(obj_dir):
            continue
        imgs = sorted([
            os.path.join(obj_dir, f)
            for f in os.listdir(obj_dir)
            if f.lower().endswith((".jpg", ".png", ".jpeg"))
        ])
        if len(imgs) == 0:
            continue
        sample_refs[s] = imgs
        base_used.add(base_name)

    print(f"[INFO] Unique object types: {len(base_used)}, dùng {len(sample_refs)} folder ref.")
    return sample_refs

# ------------ Helper: collect backgrounds ------------
def collect_train_backgrounds(train_samples_dir, forbidden_map, max_per_video=6):
    """
    Lấy background từ video TRAIN, tránh frame có object thật (theo annotations).
    + cộng thêm static .jpg/.png trong folder nếu có.
    """
    bg_paths = []
    tmp_dir = "/kaggle/working/_synth_bg_tmp_rembg"
    os.makedirs(tmp_dir, exist_ok=True)

    for folder in sorted(glob(os.path.join(train_samples_dir, "*"))):
        vid = os.path.join(folder, "drone_video.mp4")
        if not os.path.exists(vid):
            continue

        video_id         = os.path.basename(folder)
        forbidden_frames = forbidden_map.get(video_id, set())

        cap   = cv2.VideoCapture(vid)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release()
            continue

        all_idx = np.arange(total, dtype=int)
        if len(forbidden_frames) > 0:
            mask = ~np.isin(all_idx, list(forbidden_frames))
            valid_idx = all_idx[mask]
        else:
            valid_idx = all_idx

        if len(valid_idx) == 0:
            cap.release()
            continue

        k = min(max_per_video, len(valid_idx))
        chosen_positions = np.linspace(0, len(valid_idx) - 1, k, dtype=int)
        chosen_frames    = valid_idx[chosen_positions]

        for i in chosen_frames:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
            ret, frame = cap.read()
            if ret:
                p = os.path.join(tmp_dir, f"{video_id}_f{i}.jpg")
                cv2.imwrite(p, frame)
                bg_paths.append(p)

        cap.release()

    for folder in sorted(glob(os.path.join(train_samples_dir, "*"))):
        imgs = glob(os.path.join(folder, "*.jpg")) + glob(os.path.join(folder, "*.png"))
        bg_paths += imgs

    print(f"[INFO] Thu được {len(bg_paths)} background images.")
    return bg_paths

# ------------ Helper: remove bg bằng rembg (U2Net) ------------
def rembg_cutout(img_bgr):
    """
    Dùng rembg (U2Net) tách nền:
    Trả về (rgb, alpha[0..1])
    """
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    out_rgba = remove(img_rgb)  # HxWx4, RGBA
    if out_rgba is None:
        return None, None
    out_rgba = cv2.cvtColor(out_rgba, cv2.COLOR_RGBA2BGRA)
    b,g,r,a = cv2.split(out_rgba)
    rgb = cv2.merge([b,g,r])
    alpha = a.astype(np.float32) / 255.0
    return rgb, alpha

def save_rgba(img_bgr, alpha, out_path):
    b,g,r = cv2.split(img_bgr)
    a     = (alpha*255).astype(np.uint8)
    rgba  = cv2.merge([b,g,r,a])
    cv2.imwrite(out_path, rgba)

# ------------ Helper: alpha paste & bbox ------------
def alpha_paste(bg, fg_rgba, x, y):
    if fg_rgba.dtype != np.uint8:
        fg_rgba = (fg_rgba*255).astype(np.uint8)
    fg_bgr = fg_rgba[:,:,0:3]
    alpha  = fg_rgba[:,:,3].astype(float) / 255.0

    h, w = fg_bgr.shape[:2]
    H, W = bg.shape[:2]
    if x >= W or y >= H:
        return bg

    x1 = max(0, x);  y1 = max(0, y)
    x2 = min(W, x + w); y2 = min(H, y + h)
    ox1 = x1 - x;  oy1 = y1 - y
    ox2 = ox1 + (x2 - x1); oy2 = oy1 + (y2 - y1)

    alpha_patch = alpha[oy1:oy2, ox1:ox2][:,:,None]
    fg_patch    = fg_bgr[oy1:oy2, ox1:ox2].astype(float)
    bg_patch    = bg[y1:y2, x1:x2].astype(float)

    comp = alpha_patch * fg_patch + (1 - alpha_patch) * bg_patch
    bg[y1:y2, x1:x2] = comp.astype("uint8")
    return bg

def random_affine_params(img_shape, max_rot=ROT_DEG, scale_low=0.9, scale_high=1.1):
    h, w = img_shape[:2]
    angle = random.uniform(-max_rot, max_rot)
    scale = random.uniform(scale_low, scale_high)
    cx, cy = w/2, h/2
    M = cv2.getRotationMatrix2D((cx, cy), angle, scale)
    return M

def apply_affine(img, M, out_size=None, interp=cv2.INTER_LINEAR, borderValue=(0,0,0)):
    h,w = img.shape[:2]
    if out_size is None:
        out_size = (w,h)
    return cv2.warpAffine(
        img, M, out_size,
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=borderValue
    )

def sample_scale_biased(min_s, max_s, bias_exp=BIAS_EXP):
    # bias_exp < 1 -> thiên về scale lớn
    u = random.random()
    return min_s + (max_s - min_s) * (u ** bias_exp)

def sample_centered_position(W, H, obj_w, obj_h, center_margin=CENTER_MARGIN):
    pad_x = int(center_margin * W)
    pad_y = int(center_margin * H)
    min_x = pad_x
    max_x = max(pad_x, W - obj_w - pad_x)
    min_y = pad_y
    max_y = max(pad_y, H - obj_h - pad_y)

    if max_x <= min_x:
        x = min_x
    else:
        x = random.randint(min_x, max_x)
    if max_y <= min_y:
        y = min_y
    else:
        y = random.randint(min_y, max_y)
    return x, y

# ------------ Occlusion & Edge cut-off helpers ------------

def apply_partial_occlusion(img, bbox, area_range=OCCLUDE_AREA_RANGE):
    """
    Dán 1 patch hình chữ nhật giả lập vật cản che 1 phần object.
    Patch lấy từ chính background quanh bbox.
    """
    H, W = img.shape[:2]
    x1, y1, x2, y2 = bbox
    bw = x2 - x1
    bh = y2 - y1
    if bw <= 5 or bh <= 5:
        return img

    # diện tích patch ~ 10–30% diện tích bbox
    bbox_area = bw * bh
    r = random.uniform(*area_range)
    target_area = bbox_area * r

    aspect = random.uniform(0.5, 2.0)
    h = int(np.sqrt(target_area / aspect))
    w = int(aspect * h)

    h = max(3, min(h, bh))
    w = max(3, min(w, bw))

    # chọn vị trí trong bbox
    px1 = random.randint(x1, max(x1, x2 - w))
    py1 = random.randint(y1, max(y1, y2 - h))
    px2 = px1 + w
    py2 = py1 + h

    # nguồn patch: lấy từ vùng gần bbox (nhưng vẫn trong ảnh)
    sx1 = max(0, px1 - random.randint(0, 15))
    sy1 = max(0, py1 - random.randint(0, 15))
    sx2 = min(W, sx1 + w)
    sy2 = min(H, sy1 + h)

    if sx2 <= sx1 or sy2 <= sy1:
        return img

    patch = img[sy1:sy2, sx1:sx2].copy()
    ph, pw = patch.shape[:2]
    if ph <= 1 or pw <= 1:
        return img

    patch = cv2.resize(patch, (w, h), interpolation=cv2.INTER_LINEAR)
    img[py1:py2, px1:px2] = patch
    return img

def apply_edge_cutoff(img, bbox, max_cut_ratio=0.3):
    H, W = img.shape[:2]
    x1, y1, x2, y2 = bbox
    bw = x2 - x1
    bh = y2 - y1

    if bw <= 5 or bh <= 5:
        return img, bbox

    # ---- 1. Crop object từ vị trí cũ ----
    obj_crop = img[y1:y2, x1:x2].copy()
    oh, ow = obj_crop.shape[:2]

    # ---- 2. "Xoá" object ở vị trí cũ bằng patch background lân cận ----
    # Lấy patch từ phía trên hoặc dưới object để đè lên
    src_y1 = max(0, y1 - bh)
    src_y2 = min(H, src_y1 + bh)
    src_x1 = max(0, x1 - bw//2)
    src_x2 = min(W, src_x1 + bw)

    if (src_y2 - src_y1) > 5 and (src_x2 - src_x1) > 5:
        bg_patch = img[src_y1:src_y2, src_x1:src_x2].copy()
        bg_patch = cv2.resize(bg_patch, (bw, bh), interpolation=cv2.INTER_LINEAR)
        img[y1:y2, x1:x2] = bg_patch
    else:
        # fallback: blur mạnh để xoá chi tiết
        img[y1:y2, x1:x2] = cv2.GaussianBlur(img[y1:y2, x1:x2], (15, 15), 0)

    # ---- 3. Chọn cạnh để "dính" vào và cắt bớt ----
    side = random.choice(["left", "right", "top", "bottom"])
    cut_ratio = random.uniform(0.05, max_cut_ratio)

    if side in ["left", "right"]:
        cut_w = int(ow * cut_ratio)
        if cut_w >= ow:
            cut_w = ow - 1
        if side == "left":
            obj_crop = obj_crop[:, cut_w:]          # mất 1 phần bên trái
        else:  # right
            obj_crop = obj_crop[:, :ow - cut_w]     # mất 1 phần bên phải
    else:
        cut_h = int(oh * cut_ratio)
        if cut_h >= oh:
            cut_h = oh - 1
        if side == "top":
            obj_crop = obj_crop[cut_h:, :]          # mất 1 phần phía trên
        else:  # bottom
            obj_crop = obj_crop[:oh - cut_h, :]     # mất 1 phần phía dưới

    nh, nw = obj_crop.shape[:2]
    if nh <= 2 or nw <= 2:
        return img, bbox

    # ---- 4. Tính vị trí mới sát mép ảnh ----
    if side == "left":
        new_x1 = 0
        new_x2 = min(nw, W)
        cy = (y1 + y2) // 2
        new_y1 = max(0, cy - nh // 2)
        new_y2 = min(H, new_y1 + nh)
    elif side == "right":
        new_x2 = W
        new_x1 = max(0, W - nw)
        cy = (y1 + y2) // 2
        new_y1 = max(0, cy - nh // 2)
        new_y2 = min(H, new_y1 + nh)
    elif side == "top":
        new_y1 = 0
        new_y2 = min(nh, H)
        cx = (x1 + x2) // 2
        new_x1 = max(0, cx - nw // 2)
        new_x2 = min(W, new_x1 + nw)
    else:  # bottom
        new_y2 = H
        new_y1 = max(0, H - nh)
        cx = (x1 + x2) // 2
        new_x1 = max(0, cx - nw // 2)
        new_x2 = min(W, new_x1 + nw)

    # clamp + crop lại cho khớp
    new_x2 = min(new_x2, W)
    new_y2 = min(new_y2, H)
    obj_crop = obj_crop[:new_y2 - new_y1, :new_x2 - new_x1]

    img[new_y1:new_y2, new_x1:new_x2] = obj_crop
    new_bbox = (new_x1, new_y1, new_x2, new_y2)
    return img, new_bbox

def paste_obj_to_bg_with_alpha(bg, fg_rgb, alpha_mask, scale_rel,
                               alpha_thr=ALPHA_THR, erosion_iter=EROSION_ITER,
                               center_margin=CENTER_MARGIN):
    """
    - Resize fg_rgb & alpha giữ tỉ lệ
    - Dùng alpha_mask để lấy tight bbox
    - Paste object lên bg bằng alpha blending
    - Trả về (out_image, bbox) với bbox = (x1,y1,x2,y2) trên bg
    """
    H, W = bg.shape[:2]
    h0, w0 = fg_rgb.shape[:2]
    if h0 == 0 or w0 == 0:
        return None, None

    target = int(scale_rel * min(H, W))
    factor = target / max(1, min(h0, w0))
    new_w  = max(2, int(w0 * factor))
    new_h  = max(2, int(h0 * factor))

    fg_resized    = cv2.resize(fg_rgb,   (new_w, new_h), interpolation=cv2.INTER_AREA)
    alpha_resized = cv2.resize(alpha_mask, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    alpha_resized = np.clip(alpha_resized.astype(np.float32), 0.0, 1.0)

    mask_bin = (alpha_resized > alpha_thr).astype(np.uint8)
    if erosion_iter > 0:
        kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
        mask_bin = cv2.erode(mask_bin, kern, iterations=erosion_iter)

    ys, xs = np.where(mask_bin > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None, None

    x1_c, x2_c = xs.min(), xs.max()
    y1_c, y2_c = ys.min(), ys.max()

    crop_rgb   = fg_resized[y1_c:y2_c+1, x1_c:x2_c+1]
    crop_alpha = alpha_resized[y1_c:y2_c+1, x1_c:x2_c+1]
    crop_h, crop_w = crop_rgb.shape[:2]
    if crop_h <= 0 or crop_w <= 0:
        return None, None

    alpha_feather = cv2.GaussianBlur(crop_alpha.astype(np.float32), (7,7), 0)
    if alpha_feather.max() > 0:
        alpha_feather = (alpha_feather / alpha_feather.max()).clip(0,1)
    else:
        alpha_feather = crop_alpha

    a_u     = (alpha_feather * 255).astype(np.uint8)
    fg_rgba = cv2.merge([crop_rgb[:,:,0], crop_rgb[:,:,1], crop_rgb[:,:,2], a_u])

    x, y = sample_centered_position(W, H, crop_w, crop_h, center_margin=center_margin)

    out = bg.copy()
    out = alpha_paste(out, fg_rgba, x, y)

    bbox = (x, y, x + crop_w, y + crop_h)
    return out, bbox

# -------------- MAIN SYNTHETIC GENERATION --------------
print("Xây forbidden frame map từ annotations...")
forbidden_map = build_forbidden_frames_map(ANNOTATIONS_FILE)

print("Lấy UNIQUE reference object_images từ TRAIN...")
sample_refs = collect_unique_ref_samples(TRAIN_SAMPLES)
if len(sample_refs) == 0:
    raise SystemExit("Không tìm thấy object_images trong TRAIN.")

print("Thu background từ TRAIN (lọc frame có object)...")
bg_paths = collect_train_backgrounds(TRAIN_SAMPLES, forbidden_map, max_per_video=3)
if len(bg_paths) == 0:
    raise SystemExit("Không có background nào trong TRAIN.")

sample_names = list(sample_refs.keys())
if TEST_MODE:
    sample_names = sample_names[:TEST_MAX_SAMPLES]
    print("TEST_MODE ON - chỉ dùng samples:", sample_names)

for sample in sample_names:
    refs = sample_refs[sample]
    out_img_dir = os.path.join(OUT_BASE, sample, "images")
    out_lbl_dir = os.path.join(OUT_BASE, sample, "labels")
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    masked_refs = []
    print(f"Xử lý ref cho sample {sample} ...")
    for ref_p in refs:
        img = cv2.imread(ref_p)
        if img is None:
            continue
        # dùng rembg để tách nền
        rgb, alpha = rembg_cutout(img)
        if rgb is None or alpha is None:
            continue
        masked_refs.append((rgb, alpha))

        if DEBUG_SAVE_ALPHA:
            save_rgba(rgb, alpha, os.path.join(
                out_img_dir,
                os.path.basename(ref_p).rsplit(".",1)[0] + "_alpha.png"
            ))

    if len(masked_refs) == 0:
        print("Không có ref hợp lệ cho", sample, "- bỏ qua.")
        continue

    count = 0
    for i in tqdm(range(SYNTH_PER_SAMPLE), desc=f"Synth-{sample}"):
        bg_p = random.choice(bg_paths)
        bg   = cv2.imread(bg_p)
        if bg is None:
            continue

        ref_img, ref_mask = random.choice(masked_refs)

        M = random_affine_params(ref_img.shape,
                                 max_rot=ROT_DEG,
                                 scale_low=0.95,
                                 scale_high=1.05)
        ref_img_t = apply_affine(
            ref_img, M,
            out_size=(ref_img.shape[1], ref_img.shape[0]),
            interp=cv2.INTER_LINEAR,
            borderValue=(0,0,0)
        )
        ref_mask_t = apply_affine(
            (ref_mask*255).astype(np.uint8),
            M,
            out_size=(ref_mask.shape[1], ref_mask.shape[0]),
            interp=cv2.INTER_LINEAR,
            borderValue=0
        )
        ref_mask_t = (ref_mask_t.astype(np.float32) / 255.0).clip(0,1)

        # photometric augment trên object
        if random.random() < 0.9:
            factor = 1.0 + random.uniform(-0.18, 0.18)
            ref_img_t = np.clip(ref_img_t.astype(np.float32) * factor, 0, 255).astype(np.uint8)
        if random.random() < BLUR_PROB:
            k = random.choice([3,5])
            ref_img_t = cv2.GaussianBlur(ref_img_t, (k,k), 0)

        scale_rel = sample_scale_biased(MIN_SCALE, MAX_SCALE, bias_exp=BIAS_EXP)

        out_img, bbox = paste_obj_to_bg_with_alpha(
            bg, ref_img_t, ref_mask_t, scale_rel,
            alpha_thr=ALPHA_THR,
            erosion_iter=EROSION_ITER,
            center_margin=CENTER_MARGIN
        )
        if out_img is None or bbox is None:
            continue

        # --- Global photometric (KHÔNG motion blur) ---
        out_img = random_global_illumination(out_img, p=0.85)

        # --- Occlusion / Edge cut-off nhẹ cho ~15% sample ---
        if random.random() < OCCLUSION_PROB:
            if random.random() < EDGE_PROB:
                # edge cut-off
                out_img, bbox = apply_edge_cutoff(out_img, bbox, max_cut_ratio=0.3)
            else:
                # partial occlusion
                out_img = apply_partial_occlusion(out_img, bbox, area_range=OCCLUDE_AREA_RANGE)

        x1,y1,x2,y2 = bbox
        H,W = out_img.shape[:2]

        fname = f"{sample}_synth_{i:05d}"
        cv2.imwrite(os.path.join(out_img_dir, fname + ".jpg"), out_img)

        # YOLO label normalized
        xc = ((x1+x2)/2.0)/W
        yc = ((y1+y2)/2.0)/H
        bw = (x2-x1)/W
        bh = (y2-y1)/H
        with open(os.path.join(out_lbl_dir, fname + ".txt"), "w") as f:
            f.write(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

        count += 1

    print(f"Đã tạo {count} synthetic images cho sample {sample}")

print("=== Synthetic generation DONE ===")
print("Output root:", OUT_BASE)
print("→ Merge ảnh & label trong OUT_BASE vào YOLO train hoặc thêm path này vào dataset.yaml.")


Xây forbidden frame map từ annotations...
[INFO] Forbidden frame map cho 14 video.
Lấy UNIQUE reference object_images từ TRAIN...
[INFO] Unique object types: 7, dùng 7 folder ref.
Thu background từ TRAIN (lọc frame có object)...
[INFO] Thu được 42 background images.
Xử lý ref cho sample Backpack_0 ...


100%|████████████████████████████████████████| 176M/176M [00:00<00:00, 296GB/s]
Synth-Backpack_0: 100%|██████████| 700/700 [02:37<00:00,  4.44it/s]


Đã tạo 700 synthetic images cho sample Backpack_0
Xử lý ref cho sample Jacket_0 ...


Synth-Jacket_0: 100%|██████████| 700/700 [03:33<00:00,  3.28it/s]


Đã tạo 700 synthetic images cho sample Jacket_0
Xử lý ref cho sample Laptop_0 ...


...

Đã tạo 700 synthetic images cho sample MobilePhone_0
Xử lý ref cho sample Person1_0 ...


Synth-Person1_0: 100%|██████████| 700/700 [01:44<00:00,  6.67it/s]


Đã tạo 700 synthetic images cho sample Person1_0
Xử lý ref cho sample WaterBottle_0 ...


Synth-WaterBottle_0: 100%|██████████| 700/700 [02:44<00:00,  4.27it/s]

Đã tạo 700 synthetic images cho sample WaterBottle_0
=== Synthetic generation DONE ===
Output root: /kaggle/working/synth_train_rembg
→ Merge ảnh & label trong OUT_BASE vào YOLO train hoặc thêm path này vào dataset.yaml.
